In [ ]:
import pandas as pd
import numpy as np

from alphagenome.data import gene_annotation
from alphagenome.data import genome
from alphagenome.data import transcript as transcript_utils
from alphagenome.interpretation import ism
from alphagenome.models import dna_client
from alphagenome.models import variant_scorers
from alphagenome.visualization import plot_components
import matplotlib.pyplot as plt

In [ ]:
gtf = pd.read_feather(
    'https://storage.googleapis.com/alphagenome/reference/gencode/'
    'hg38/gencode.v46.annotation.gtf.gz.feather'
)

# Set up transcript extractors using the information in the GTF file.
gtf_transcripts = gene_annotation.filter_protein_coding(gtf)
gtf_transcripts = gene_annotation.filter_to_longest_transcript(gtf_transcripts)
transcript_extractor = transcript_utils.TranscriptExtractor(gtf_transcripts)

In [ ]:
# get model
dna_model = dna_client.create("")

In [ ]:
# available outputs
[output.name for output in dna_client.OutputType]

In [ ]:
# data ontologies
output_metadata = dna_model.output_metadata(
    dna_client.Organism.HOMO_SAPIENS
).concatenate()
print(output_metadata)

In [ ]:
#hp
hp_mask = output_metadata['biosample_name'].fillna('').str.contains('hippocampus', case=False, regex=True)
hp_rows = output_metadata[hp_mask]
print("HP",hp_rows['output_type'].value_counts())

In [ ]:
# get all terms
hp_ontology = hp_rows[hp_rows['output_type'].isin([dna_client.OutputType.CHIP_HISTONE])]
hp_ontology = hp_ontology['ontology_curie']

In [ ]:
# 1:21176231:TGCC>T
variant_string = "chr1:21176231:TGCC>T"
variant = genome.Variant.from_str(variant_string)
interval = variant.reference_interval.resize(dna_client.SEQUENCE_LENGTH_1MB)

variant_output = dna_model.predict_variant(
    interval=interval,
    variant=variant,
    requested_outputs=[
                       dna_client.OutputType.CHIP_HISTONE,
                       ],
    ontology_terms=hp_ontology,
) 

longest_transcripts = transcript_extractor.extract(interval)
gene_interval1 = gene_annotation.get_gene_interval(gtf, gene_symbol='EIF4G3')
gene_interval1.resize_inplace(gene_interval1.width + 300000)

In [ ]:
npg_colors = [
    "#E64B35",
    "#4DBBD5",
    "#00A087",
    "#3C5488",
    "#F39B7F",
    "#8491B4",
    "#91D1C2",
    "#DC0000",
    "#7E6148",
    "#B09C85"
]

#difference plot
fig = plot_components.plot(
    [
        plot_components.TranscriptAnnotation(longest_transcripts),

        plot_components.Tracks(
            tdata=variant_output.alternate.chip_histone
            - variant_output.reference.chip_histone,
            ylabel_template='{biosample_name}\n{name}',
            filled=True,
        ),
    ],
    annotations=[plot_components.VariantAnnotation([variant], alpha=0.5, label_angle=0)],
    interval=variant_output.reference.chip_histone.interval.resize(2**15),
    title=(
        'Predicted difference of chr1:21176231:TGCC>T'
    )
)

fig.set_size_inches(4, 3.5)
ax = fig.gca()
i = 0
for ax in fig.axes:
    for poly in ax.collections:     
        poly.set_facecolor(npg_colors[i % len(npg_colors)])
        poly.set_edgecolor(npg_colors[i % len(npg_colors)])
        i += 1
    
plt.show()
fig.savefig('Plots/chr1_21176231_TGCC_T_EIF4G3.pdf', dpi=600, bbox_inches='tight')

In [ ]:
# chr3:183642919:A>G: KLHL24
variant_string = "chr3:183642919:A>G"
variant = genome.Variant.from_str(variant_string)
interval = variant.reference_interval.resize(dna_client.SEQUENCE_LENGTH_1MB)

variant_output = dna_model.predict_variant(
    interval=interval,
    variant=variant,
    requested_outputs=[
                       dna_client.OutputType.CHIP_HISTONE
                       ],
    ontology_terms=hp_ontology,
) 

longest_transcripts = transcript_extractor.extract(interval)
gene_interval = gene_annotation.get_gene_interval(gtf, gene_symbol='KLHL24')
gene_interval.resize_inplace(gene_interval.width + 300000)

In [ ]:
#difference plot
fig = plot_components.plot(
    [
        plot_components.TranscriptAnnotation(longest_transcripts),

        plot_components.Tracks(
            tdata=variant_output.alternate.chip_histone
            - variant_output.reference.chip_histone,
            ylabel_template='{biosample_name}\n{name}',
            filled=True,
        ),
    ],
    annotations=[plot_components.VariantAnnotation([variant], alpha=0.5, label_angle=0)],
    interval=variant_output.reference.chip_histone.interval.resize(2**14),
    title=(
        'Predicted difference of chr3:183642919:A>G'
    )
)

fig.set_size_inches(4, 3.5)
ax = fig.gca()
i = 0
for ax in fig.axes:
    for poly in ax.collections:     
        poly.set_facecolor(npg_colors[i % len(npg_colors)])
        poly.set_edgecolor(npg_colors[i % len(npg_colors)])
        i += 1
    
plt.show()
fig.savefig('Plots/chr3_183642919_A_G_KLHL24.pdf', dpi=600, bbox_inches='tight')